# Jupyter notebook for rocprof-insights

## run rocprof (v1) as
$ mkdir -p v1_prof

$ rocprof --timestamp on --hsa-trace --hip-trace  --sys-trace -o ./v1_prof/v1_prof.csv python3 training.py

## collect data via rocprof (v1)

In [ ]:
!rm -rf v1_prof
!mkdir -p v1_prof
!rocprof --timestamp on --hsa-trace --hip-trace  --sys-trace -o ./v1_prof/v1_prof.csv python3 training.py

In [ ]:
import os
import pandas as pd
from rocprof_insights.rocloader import RocprofLoader
from rocprof_insights.rocanalysis import RocAnalyzer
from rocprof_insights.rocvisuals import RocprofStatsVisualizer, MemoryCopyVisualizer

In [ ]:
!ls -hltar ./v1_prof

## Load kernel trace data

### Kernel latency

In [ ]:
# adapt to the local path of the kernel trace file
kernel_file_path = './v1_prof/v1_prof.csv'

In [ ]:
df_raw = pd.read_csv(kernel_file_path)
df_raw.head(10)

In [ ]:
dloader = RocprofLoader(kernel_file_path, 'v1')
df = dloader.load_data()

In [ ]:
df.columns

In [ ]:
kernel_names = list(df['kernel_name'])
max_length = max(len(x) for x in kernel_names)
print(f'max_length of kernel name = {max_length}') 

In [ ]:
rocAnalyzer = RocAnalyzer(df, required_cols={'kernel_name', 'duration_us', 'Private_Segment_Size [Kb]', 'Group_Segment_Size [Kb]'})

In [ ]:
stats_df = rocAnalyzer.compute_advanced_stats()

In [ ]:
stats_df.head(10)

In [ ]:
roc_viz = RocprofStatsVisualizer(stats_df)

In [ ]:
roc_viz.histogram_a_field(field = 'total time [s]', title = "Histogram of Total Duration")

In [ ]:
roc_viz.pie_chart_top10_a_field(field='kernel_name', sort_field='total time [s]')

In [ ]:
roc_viz.assign_categories_and_plot()

In [ ]:
# roc_viz.box_plot_time()

In [ ]:
roc_viz.scatter_avg_vs_num_calls(field='kernel_name')

### Kernel configurations

In [ ]:
#### local_memory to check register spilling 

In [ ]:
roc_viz.histogram_a_field(df=df, field = 'Private_Segment_Size [Kb]', title = "Histogram of local memory")

In [ ]:
roc_viz.histogram_a_field(df=df, field = 'Group_Segment_Size [Kb]', title = "Histogram of Group_Segment_Size")